
## 1. Introducción

Esta libreta simula la **llegada incremental de transacciones y etiquetas de fraude** al sistema de producción, replicando el comportamiento de un entorno real donde los datos llegan de forma continua desde sistemas externos.

El mecanismo consiste en leer los archivos `data.json` almacenados en la carpeta **`source_buffer`** bajo el volumen **`landing_zone`**, que contienen los datos del futuro aún no procesados, y escribir sus registros en la estructura de directorios de **`events`**, respetando la **partición por año y mes**. Cada lote se escribe como un fichero independiente, de forma que el **`Auto Loader`** detecta cada fichero nuevo y lo ingiere de forma incremental sin necesidad de modificar ficheros existentes.

Las transacciones se copian en **orden cronológico estricto**. Para cada ventana de tiempo procesada, se copian también todas las etiquetas cuyo **`label_available_date`** cae dentro de esa misma ventana, simulando el **retraso real** con el que los equipos de revisión confirman los casos de fraude.

La cadencia de la simulación se controla mediante dos parámetros configurables:

* **`rows_per_batch`**: número de filas de transacciones que se copian en cada iteración del bucle.
* **`seconds_per_batch`**: segundos de espera entre iteraciones.

La combinación de ambos permite expresar cualquier cadencia deseada. Por ejemplo, `rows_per_batch = 100` con `seconds_per_batch = 5` copia **100 transacciones cada 5 segundos**. Si el procesamiento de un lote tarda más de `seconds_per_batch` segundos, el bucle **continúa inmediatamente** sin esperar.

La simulación es **idempotente**: antes de comenzar, localiza automáticamente el **`timestamp` máximo** ya presente en `events/transactions` y omite todas las filas anteriores o iguales a ese valor, por lo que puede relanzarse sin duplicar datos en caso de fallo.


## 1. Importaciones y configuración

In [0]:
exec(open("07_Utils.py").read(), globals())

In [0]:
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

In [0]:
# Base paths on the volume
landing_zone_path = Path("/") / "Volumes"/ catalog / database / "landing_zone"
source_buffer_tx_path = landing_zone_path / "source_buffer" / "transactions"
source_buffer_lbl_path = landing_zone_path / "source_buffer" / "labels"
events_tx_path = landing_zone_path / "events" / "transactions"
events_lbl_path = landing_zone_path / "events" / "labels"

# Number of rows to copy in each batch
rows_per_batch = 5000

# Seconds to wait between batches
seconds_per_batch = 5

print(f"Source buffer (transactions): {source_buffer_tx_path}")
print(f"Source buffer (labels): {source_buffer_lbl_path}")
print(f"Events (transactions): {events_tx_path}")
print(f"Events (labels): {events_lbl_path}")
print(f"Rows per batch : {rows_per_batch}")
print(f"Seconds per batch: {seconds_per_batch}")


## 2. Punto de continuación

Antes de comenzar la simulación se determina desde qué punto debe continuar, consultando el `timestamp` máximo ya presente en `events/transactions` mediante una lectura distribuida con **`Spark`**. Todas las transacciones del *buffer* con `timestamp` estrictamente posterior a ese valor serán las candidatas a copiar.

In [0]:
def _find_latest_events_timestamp():
    """
    Query the maximum `timestamp` already present in `events/transactions`
    and return it as a `datetime` object.

    Returns `None` when no transactions have been copied yet.
    """
    try:
        max_ts = (
            spark.read
                 .json(str(events_tx_path / "*" / "*" / "*.json"))
                 .agg(F.max("timestamp"))
                 .first()[0]
        )
        return datetime.fromisoformat(max_ts) if max_ts else None
    except Exception:
        return None


resume_from = _find_latest_events_timestamp()

if resume_from is None:
    print("No prior transactions found. Simulation will start from the beginning of the buffer.")
else:
    print(f"Resuming from timestamp: {resume_from.isoformat()}")
    print("Only transactions strictly after this timestamp will be copied.")


## 3. Carga y ordenación del *buffer* de transacciones

Se leen todos los archivos `data.json` de `source_buffer/transactions` mediante una lectura distribuida con **`Spark`**, se fusionan en una única lista ordenada cronológicamente por **`timestamp`** y se filtran las filas ya procesadas según el punto de continuación determinado en la sección anterior.

In [0]:
def _load_buffer(base_path):
    """
    Extract `_year` and `_month` from the partition path of each record
    and return the full buffer as a flat list of dicts for in-memory processing.
    """
    return (
        spark.read
             .json(str(base_path / "*" / "*" / "data.json"))
             .withColumn("_year",  F.element_at(F.split(F.col("_metadata.file_path"), "/"), -3))
             .withColumn("_month", F.element_at(F.split(F.col("_metadata.file_path"), "/"), -2))
             .toPandas()
             .to_dict("records")
    )


all_tx = _load_buffer(source_buffer_tx_path)
for row in all_tx:
    row["_ts"] = datetime.fromisoformat(str(row["timestamp"]))
all_tx.sort(key = lambda row: row["_ts"])
print(f"Total rows in buffer : {len(all_tx):,}")

if resume_from is not None:
    pending_tx = [row for row in all_tx if row["_ts"] > resume_from]
else:
    pending_tx = all_tx

print(f"Rows already in events: {len(all_tx) - len(pending_tx):,}")
print(f"Rows pending simulation: {len(pending_tx):,}")


## 4. Carga del *buffer* de etiquetas

Se leen todos los archivos `data.json` de `source_buffer/labels` con la misma estrategia que las transacciones. Solo se consideran las etiquetas cuyo **`label_available_date`** no es nulo, ya que las restantes corresponden a casos aún no resueltos por los equipos de revisión.

In [0]:
all_lbl = _load_buffer(source_buffer_lbl_path)
print(f"Total rows in label buffer: {len(all_lbl):,}")

for row in all_lbl:
    lad = row.get("label_available_date")
    row["_lad"] = datetime.fromisoformat(lad) if lad else None

available_lbl = [row for row in all_lbl if row["_lad"] is not None]
available_lbl.sort(key = lambda row: row["_lad"])

# Track which labels have already been copied
copied_label_ids = set()

print(f"Labels with valid available date: {len(available_lbl):,}")
print(f"Labels with null available date (skipped): {len(all_lbl) - len(available_lbl):,}")


## 5. Bucle principal de simulación

El bucle procesa las transacciones pendientes en lotes de **`rows_per_batch`** filas, esperando **`seconds_per_batch`** segundos entre cada iteración. Cada lote define una **ventana temporal** delimitada por el `timestamp` de su primera y última fila, que se usa para determinar qué etiquetas deben copiarse en ese mismo ciclo. Cada lote se escribe como un **fichero independiente** en *newline-delimited* `.json`, de forma que el **`Auto Loader`** lo detecta y lo ingiere sin necesidad de modificar ficheros existentes.

In [0]:
def _write_json(dest_path, records):
    """
    Write `records` to `dest_path` on the volume in newline-delimited
    `.json`s format (one record per line).
    """
    lines = "\n".join(json.dumps(record, default = str) for record in records)
    dbutils.fs.put(dest_path, lines, overwrite = True)


def _clean_row(row):
    """
    Remove internal metadata keys added during loading before writing to disk.
    """
    return {key: value for key, value in row.items() if not key.startswith("_")}


### 5.1. Copia de etiquetas por ventana temporal

Por cada lote de transacciones, se seleccionan las etiquetas cuyo **`label_available_date`** cae dentro de la ventana temporal del lote y que aún no han sido copiadas en iteraciones anteriores. Las etiquetas se agrupan por partición de destino `(year, month)` y se escriben en **`events/labels`** como un fichero independiente por lote.

In [0]:
def _copy_labels(window_start, window_end, batch_number):
    """
    Copy all labels whose `label_available_date` falls within
    `[window_start, window_end]` and have not yet been copied.

    Returns the number of labels written.
    """
    to_copy = [
        row for row in available_lbl
        if window_start <= row["_lad"] <= window_end
        and row["transaction_id"] not in copied_label_ids
    ]
    if not to_copy:
        return 0

    # Group by destination partition (year, month)
    partitions = {}
    for row in to_copy:
        key = (row["_year"], row["_month"])
        partitions.setdefault(key, []).append(row)

    for (year, month), rows in partitions.items():
        dest_path = str(events_lbl_path / year / month / f"batch_{batch_number:06d}.json")
        _write_json(dest_path, [_clean_row(r) for r in rows])
        for row in rows:
            copied_label_ids.add(row["transaction_id"])

    return len(to_copy)


### 5.2. Ejecución

Las transacciones de cada lote se agrupan por partición de destino `(year, month)` y se escriben en **`events/transactions`** como un fichero `.json` independiente.

In [0]:
if not pending_tx:
    print("No pending transactions. The buffer is fully consumed.")
else:
    total_tx_copied  = 0
    total_lbl_copied = 0
    total_batches = 0
    n_pending = len(pending_tx)

    header = f"Starting simulation: {n_pending:,} transactions to copy, {rows_per_batch} rows every {seconds_per_batch} second(s)."
    separator = "-" * len(header)

    print(header)
    print(separator)

    batch_start = 0

    while batch_start < n_pending:
        t0 = time.time()
        batch = pending_tx[batch_start : batch_start + rows_per_batch]

        window_start = batch[0]["_ts"]
        window_end = batch[-1]["_ts"]
        total_batches += 1

        # 1. Copy transactions, grouped by (year, month) partition
        tx_partitions = {}
        for row in batch:
            key = (row["_year"], row["_month"])
            tx_partitions.setdefault(key, []).append(row)

        for (year, month), rows in tx_partitions.items():
            dest_path = str(events_tx_path / year / month / f"batch_{total_batches:06d}.json")
            _write_json(dest_path, [_clean_row(r) for r in rows])

        # 2. Copy labels whose available date is in the window
        n_lbl = _copy_labels(window_start, window_end, total_batches)

        total_tx_copied += len(batch)
        total_lbl_copied += n_lbl
        batch_start += rows_per_batch

        elapsed = time.time() - t0
        print(
            f"Batch {total_batches:>4d} | "
            f"transactions copied: {len(batch):>4d} ({total_tx_copied:>7,} total) | "
            f"labels copied: {n_lbl:>4d} ({total_lbl_copied:>6,} total) | "
            f"window: {window_start.strftime('%Y-%m-%d %H:%M:%S')} → {window_end.strftime('%Y-%m-%d %H:%M:%S')} | "
            f"elapsed: {elapsed:.2f} seconds"
        )

        # Sleep for the remainder of the interval if processing was faster
        sleep_time = max(0.0, seconds_per_batch - elapsed)
        if sleep_time > 0 and batch_start < n_pending:
            time.sleep(sleep_time)

    print(separator)
    print("Simulation complete.")
    print(f"Transactions copied: {total_tx_copied:,}")
    print(f"Labels copied: {total_lbl_copied:,}")
    print(f"Batches processed: {total_batches:,}")


## 6. Conclusiones y siguientes pasos

### ¿Qué hace esta libreta?

1. **Continuación idempotente**: Antes de comenzar, consulta el `timestamp` máximo ya presente en `events/transactions` mediante una lectura distribuida con `Spark`, omitiendo todas las filas anteriores o iguales a ese valor para evitar duplicados en ejecuciones repetidas.
2. **Copia ordenada y particionada**: Las transacciones se procesan en **orden cronológico estricto** y se escriben en `events` respetando la partición `year/month`, generando un fichero independiente por lote que el **`Auto Loader`** detecta e ingiere de forma incremental.
3. **Propagación simultánea de etiquetas**: Por cada lote de transacciones, se copian las etiquetas cuyo **`label_available_date`** cae dentro de la ventana temporal del lote, simulando el retraso real de confirmación de fraude.
4. **Cadencia configurable**: Los parámetros **`rows_per_batch`** y **`seconds_per_batch`** controlan la velocidad de la simulación de forma independiente sin modificar ninguna otra lógica.

### ¿Cuándo ejecutar esta libreta?

Esta libreta se ejecuta **manualmente** desde el entorno de desarrollo para poblar `events` con datos nuevos. Una vez que el **`Auto Loader`** detecte los nuevos ficheros, la cadena completa `Run_Medallion_Pipeline → Publish_to_Online_Store → Run_Inference_And_Label_Enrichment` se encargará de procesar los datos de extremo a extremo.

### ¿Qué sigue?

Tras cada ejecución, lanza manualmente el trabajo `Credit Card Fraud Feature Pipeline` (o espera a que el *scheduler* lo dispare) para que el *pipeline* `Medallion` ingiera los nuevos ficheros y el modelo `champion` genere predicciones sobre las transacciones recién llegadas.